<a href="https://colab.research.google.com/github/nilesh-salpe/genai/blob/main/notebooks/BPE_Explained.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Byte Pair Encoding (BPE) — Explained with Two English Examples

**Byte Pair Encoding (BPE)** is a tokenization algorithm used by many modern language models (like GPT and others) to break words into smaller, reusable pieces called **subword tokens**.

### Why not just split on whole words?
- A vocabulary of *all* whole words would be huge, and the model would still fail on new/unseen words (like `"unhappiness"` if it only ever saw `"happy"`).

### Why not just split into single characters?
- That gives a tiny vocabulary, but sequences become very long and the model loses the notion of common word chunks (like `"ing"`, `"est"`, `"un"`).

**BPE is the middle ground.** It starts with individual characters and repeatedly merges the *most frequently occurring pair* of symbols into a single new symbol — building up common subwords step by step, purely by counting statistics in a text corpus.

In this notebook, we will:
1. Build BPE **from scratch in plain Python** (no libraries needed) so every step is visible.
2. Run it on **Example 1**: a tiny toy corpus, and watch the merge rules get learned one by one.
3. Use those learned merge rules on **Example 2**: brand-new words the algorithm has never seen, to see how BPE tokenizes them.

> This notebook is self-contained and will run top-to-bottom in Google Colab with no extra installs.

## Step 0: The BPE Algorithm in Plain English

1. Split every word in the training corpus into individual **characters**, and mark the end of each word with a special symbol `</w>` (so the model can tell `"low"` apart from `"lower"`).
2. Count how often every **adjacent pair of symbols** occurs across the whole corpus (weighted by word frequency).
3. Find the **single most frequent pair** and merge it everywhere into one new symbol.
4. Repeat steps 2–3 for a fixed number of merges (or until no pairs are left).
5. The list of merges you learned, in order, **is your BPE tokenizer**. To tokenize a brand-new word later, you just apply the same merges, in the same order, to that new word's characters.

## Example 1: Learning BPE Merges from a Toy Corpus

We'll use a small, classic-style training corpus of English words with made-up frequencies (how many times each word appeared in some text). Watch how words that share endings like `newest` and `widest` end up sharing subword tokens.

In [ ]:
# --- Step 1: Define our training corpus ---
# Each key is a word, each value is how many times it occurred in our "text".
# (In a real system this would be counted from millions of words of text.)
corpus = {
    "low": 5,
    "lower": 2,
    "newest": 6,
    "widest": 3,
}

# --- Step 2: Break every word into characters, plus an end-of-word marker ---
# The "</w>" marker matters: without it, BPE couldn't tell where one word
# ends and another begins once symbols start getting merged together.
def word_to_symbols(word):
    return list(word) + ["</w>"]

# vocab maps: tuple-of-symbols -> frequency
# e.g. ('l', 'o', 'w', '</w>') -> 5
vocab = {tuple(word_to_symbols(word)): freq for word, freq in corpus.items()}

print("Starting vocabulary (every word split into individual characters):")
for word_symbols, freq in vocab.items():
    print(f"  {word_symbols}   (frequency={freq})")


Starting vocabulary (every word split into individual characters):
  ('l', 'o', 'w', '</w>')   (frequency=5)
  ('l', 'o', 'w', 'e', 'r', '</w>')   (frequency=2)
  ('n', 'e', 'w', 'e', 's', 't', '</w>')   (frequency=6)
  ('w', 'i', 'd', 'e', 's', 't', '</w>')   (frequency=3)


In [ ]:
# --- Step 3: Helper functions for BPE ---
from collections import defaultdict

def get_pair_frequencies(vocab):
    """
    Look at every word in the vocabulary and count how many times each
    ADJACENT pair of symbols occurs, weighted by that word's frequency.

    Example: if ('n','e','w','e','s','t','</w>') has frequency 6,
    then the pair ('e','w') gets +6, the pair ('w','e') gets +6, etc.
    """
    pair_counts = defaultdict(int)
    for symbols, freq in vocab.items():
        for i in range(len(symbols) - 1):
            pair = (symbols[i], symbols[i + 1])
            pair_counts[pair] += freq
    return pair_counts


def merge_pair_in_vocab(pair_to_merge, vocab_in):
    """
    Go through every word in the vocabulary and merge every occurrence
    of `pair_to_merge` into one new combined symbol.

    Example: merging ('e', 's') turns ('n','e','w','e','s','t','</w>')
    into ('n','e','w','es','t','</w>').
    """
    merged_symbol = "".join(pair_to_merge)
    vocab_out = {}
    for symbols, freq in vocab_in.items():
        new_symbols = []
        i = 0
        while i < len(symbols):
            # If the current position matches the pair we're merging, combine them
            if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair_to_merge:
                new_symbols.append(merged_symbol)
                i += 2  # skip both symbols we just merged
            else:
                new_symbols.append(symbols[i])
                i += 1
        vocab_out[tuple(new_symbols)] = freq
    return vocab_out

print("Helper functions defined: get_pair_frequencies() and merge_pair_in_vocab()")


Helper functions defined: get_pair_frequencies() and merge_pair_in_vocab()


In [ ]:
# --- Step 4: Run BPE merges, one at a time, and print what happens ---

num_merges = 8          # how many merge rules we want to learn
merge_history = []      # the ordered list of merge rules BPE learns
current_vocab = dict(vocab)  # start from the character-level vocabulary

for step in range(1, num_merges + 1):
    pair_counts = get_pair_frequencies(current_vocab)
    if not pair_counts:
        print("No more pairs left to merge — stopping early.")
        break

    # Pick the single most frequent adjacent pair across the whole corpus
    best_pair = max(pair_counts, key=pair_counts.get)
    best_pair_freq = pair_counts[best_pair]

    # Apply that merge everywhere in the vocabulary
    current_vocab = merge_pair_in_vocab(best_pair, current_vocab)
    merge_history.append(best_pair)

    print(f"Merge #{step}: {best_pair[0]!r} + {best_pair[1]!r}  ->  "
          f"{''.join(best_pair)!r}   (seen together {best_pair_freq} times)")

print("\nFinal vocabulary after all merges:")
for symbols, freq in current_vocab.items():
    print(f"  {symbols}   (frequency={freq})")


Merge #1: 'e' + 's'  ->  'es'   (seen together 9 times)
Merge #2: 'es' + 't'  ->  'est'   (seen together 9 times)
Merge #3: 'est' + '</w>'  ->  'est</w>'   (seen together 9 times)
Merge #4: 'l' + 'o'  ->  'lo'   (seen together 7 times)
Merge #5: 'lo' + 'w'  ->  'low'   (seen together 7 times)
Merge #6: 'n' + 'e'  ->  'ne'   (seen together 6 times)
Merge #7: 'ne' + 'w'  ->  'new'   (seen together 6 times)
Merge #8: 'new' + 'est</w>'  ->  'newest</w>'   (seen together 6 times)

Final vocabulary after all merges:
  ('low', '</w>')   (frequency=5)
  ('low', 'e', 'r', '</w>')   (frequency=2)
  ('newest</w>',)   (frequency=6)
  ('w', 'i', 'd', 'est</w>')   (frequency=3)


### What just happened?

Notice the very first merge is almost always `('e', 's') -> 'es'`, because **both** `"newest"` (freq 6) and `"widest"` (freq 3) contain `"es"` — giving it a combined count of 9, higher than any other pair.

Soon after, you should see `'es' + 't' -> 'est'`, then eventually `'est' + '</w>' -> 'est</w>'` — BPE has discovered, purely from counting, that `"est"` is a common and meaningful English suffix, **without ever being told about English grammar**.

This ordered list of merges (`merge_history`) *is* our trained BPE tokenizer. Let's now use it on words it has never seen before.

## Example 2: Using the Learned Merges on Brand-New Words

A tokenizer isn't useful if it only works on the exact words it was trained on. The real test is: **can it sensibly break down a new word using the merge rules it already learned?**

We'll tokenize two English words that were **not** in our training corpus: `"lowest"` and `"newer"`.

In [ ]:
# --- Step 5: Apply learned BPE merges to new, unseen words ---

def apply_bpe(word, merge_rules):
    """
    Tokenize a brand-new word using an already-learned list of merge rules.
    We apply the merges in the SAME ORDER they were learned, because later
    merges often depend on earlier ones having already happened.
    """
    symbols = list(word) + ["</w>"]

    for pair in merge_rules:
        new_symbols = []
        i = 0
        while i < len(symbols):
            if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair:
                new_symbols.append("".join(pair))
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        symbols = new_symbols

    return symbols


test_words = ["lowest", "newer"]

for word in test_words:
    tokens = apply_bpe(word, merge_history)
    print(f'"{word}"  ->  tokens: {tokens}')


"lowest"  ->  tokens: ['low', 'est</w>']
"newer"  ->  tokens: ['new', 'e', 'r', '</w>']


### Reading the result

- `"lowest"` was never in the training corpus, but our BPE tokenizer already learned the chunk `"est</w>"` from `"newest"` and `"widest"`. So `"lowest"` gets split into something like `['l', 'o', 'w', 'est</w>']` — reusing a subword it already knows, instead of treating the whole word as unknown.
- `"newer"` shares the `"new"`-ish characters and the `"e"`/`"r"` characters with words BPE has seen, so it will reuse whatever merges apply, and fall back to individual characters for the rest.

This is the core superpower of BPE: **frequent chunks get merged into single tokens, rare/unseen chunks safely fall back to characters** — so the tokenizer never completely fails on a new word, and common patterns (like English suffixes `-est`, `-ing`, `-ly`, prefixes `un-`, etc.) get compact, reusable representations.

## Summary

| Concept | What it means here |
|---|---|
| **Symbol** | A character, or a previously-merged chunk of characters |
| **Pair frequency** | How often two adjacent symbols appear together across the corpus |
| **Merge rule** | "Combine this specific pair into one symbol" — learned in order, most-frequent-first |
| **`</w>`** | Marks the end of a word so merges don't accidentally cross word boundaries |
| **Trained tokenizer** | Just the ordered list of merge rules (`merge_history`) |

### Try it yourself
- Change `num_merges` in Example 1 to a bigger number (e.g. 15) and see what new merges appear.
- Add more words to `corpus`, like `"newer"`, `"slowest"`, or `"biggest"`, and re-run — watch how the learned merges change.
- Try `apply_bpe()` on a completely different English word, like `"unwidest"` (not a real word!) or your own name, and see how it gets broken down.

This is a simplified version of the algorithm — real-world tokenizers (like the ones used in GPT models) run this same process on **billions of words** and keep thousands of merge rules, but the core idea is exactly what you just built.

## Actual Usecases & Frameworks

In [ ]:
import tiktoken

for name in ["cl100k_base", "o200k_base"]:
    enc = tiktoken.get_encoding(name)
    samples = [
        "tokenization",
        "unhappiness",
        "lowest newest",
        "বাংলা",
        "👋",
    ]

    print(f"\n--- {name} (vocab size: {enc.n_vocab}) ---")
    print(f"{'String':<20} | {'Tokens':<6} | {'Pieces'}")
    print("-" * 50)

    for s in samples:
        ids = enc.encode(s)
        pieces = [enc.decode([i]) for i in ids]
        # Using repr(s) to make spaces and special characters visible
        print(f"{repr(s):<20} | {len(ids):<6} | {pieces}")


--- cl100k_base (vocab size: 100277) ---
String               | Tokens | Pieces
--------------------------------------------------
'tokenization'       | 2      | ['token', 'ization']
'unhappiness'        | 3      | ['un', 'h', 'appiness']
'lowest newest'      | 2      | ['lowest', ' newest']
'বাংলা'              | 7      | ['�', '�', 'া�', '�', '�', '�', 'া']
'👋'                  | 3      | ['�', '�', '�']

--- o200k_base (vocab size: 200019) ---
String               | Tokens | Pieces
--------------------------------------------------
'tokenization'       | 2      | ['token', 'ization']
'unhappiness'        | 3      | ['un', 'h', 'appiness']
'lowest newest'      | 2      | ['lowest', ' newest']
'বাংলা'              | 2      | ['বাংল', 'া']
'👋'                  | 2      | ['�', '�']


In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer
import os

# Create a dummy corpus file for training
CORPUS_PATH = "tiny_corpus.txt"
with open(CORPUS_PATH, "w") as f:
    f.write("low lower newest widest lowest newer unhappiness tokenization")

hf_bpe = Tokenizer(BPE(unk_token="[UNK]"))
hf_bpe.pre_tokenizer = Whitespace()
trainer = BpeTrainer(
    special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]"],
    vocab_size=120,
    min_frequency=1,
    show_progress=False,
)
hf_bpe.train([CORPUS_PATH], trainer)

# Encode a test string
test_str = "lower newest unhappiness tokenization"
encoded = hf_bpe.encode(test_str)

print(f"--- HuggingFace BPE (vocab size: {hf_bpe.get_vocab_size()}) ---")
print(f"Input: {test_str}")
print(f"{'Token':<15} | {'ID':<5}")
print("-" * 25)
for token, token_id in zip(encoded.tokens, encoded.ids):
    print(f"{token:<15} | {token_id:<5}")

--- HuggingFace BPE (vocab size: 54) ---
Input: lower newest unhappiness tokenization
Token           | ID   
-------------------------
lower           | 45   
newest          | 46   
unhappiness     | 52   
tokenization    | 53   


In [ ]:
why = [ord(x)for x in "Today, I want to start my day with a cup of coffee"]
why

[84,
 111,
 100,
 97,
 121,
 44,
 32,
 73,
 32,
 119,
 97,
 110,
 116,
 32,
 116,
 111,
 32,
 115,
 116,
 97,
 114,
 116,
 32,
 109,
 121,
 32,
 100,
 97,
 121,
 32,
 119,
 105,
 116,
 104,
 32,
 97,
 32,
 99,
 117,
 112,
 32,
 111,
 102,
 32,
 99,
 111,
 102,
 102,
 101,
 101]

In [ ]:
text = "Today, I want to start my day with a cup of coffee"

result = [(char, ord(char)) for char in text]

for char, token_id in result:
  print(f"Character: {char}, Token ID: {token_id}")

Character: T, Token ID: 84
Character: o, Token ID: 111
Character: d, Token ID: 100
Character: a, Token ID: 97
Character: y, Token ID: 121
Character: ,, Token ID: 44
Character:  , Token ID: 32
Character: I, Token ID: 73
Character:  , Token ID: 32
Character: w, Token ID: 119
Character: a, Token ID: 97
Character: n, Token ID: 110
Character: t, Token ID: 116
Character:  , Token ID: 32
Character: t, Token ID: 116
Character: o, Token ID: 111
Character:  , Token ID: 32
Character: s, Token ID: 115
Character: t, Token ID: 116
Character: a, Token ID: 97
Character: r, Token ID: 114
Character: t, Token ID: 116
Character:  , Token ID: 32
Character: m, Token ID: 109
Character: y, Token ID: 121
Character:  , Token ID: 32
Character: d, Token ID: 100
Character: a, Token ID: 97
Character: y, Token ID: 121
Character:  , Token ID: 32
Character: w, Token ID: 119
Character: i, Token ID: 105
Character: t, Token ID: 116
Character: h, Token ID: 104
Character:  , Token ID: 32
Character: a, Token ID: 97
Characte

In [ ]:
text = "This is some text"
byte_ary = bytearray(text, "utf-8")
print(byte_ary)

bytearray(b'This is some text')


In [ ]:
ids = list(byte_ary)
print(ids)

[84, 104, 105, 115, 32, 105, 115, 32, 115, 111, 109, 101, 32, 116, 101, 120, 116]


In [ ]:
print("Number of characters:", len(text))
print("Number of token IDs:", len(ids))

Number of characters: 17
Number of token IDs: 17
